In [1]:
import math
import os
import random
import re
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.stats import spearmanr
from tqdm.auto import tqdm

# Walk up from the notebook folder to locate the repository's `code/` root
CODE_DIR = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if all((p / d).is_dir() for d in ("kev", "data", "training"))),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/, and training/ at or above {Path.cwd()}")

# Run from `code/` and expose both `code/` and `code/kev/` on sys.path
os.chdir(CODE_DIR)
for _p in (str(CODE_DIR), str(CODE_DIR / "kev")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from data.base import (
    build_long_records,
    fused_eval_context,
    group_records_by_state,
    make_grouped_view,
    make_view,
    predict_probs_grouped,
)
from data.Kriner2026 import (
    AMCE_TERMS as KRINER_AMCE_TERMS,
    KRINER2026_PERSONA_COLS,
    evaluate_kriner2026_ols,
    load_kriner2026_df,
    parse_flight_description,
)
from data.Kriner2026.stimulus import CHOICE_OPTIONS as KRINER_CHOICE_OPTIONS
from data.wu2025 import WU2025_PERSONA_COLS, load_wu2025_df
from data.wu2025.stimulus import ATTR_LOCAL, ATTR_MONEY, ATTR_PERCENT, CONJOINT_OPTIONS
from models.kev_pointer import KevPointerModel
from training import free_gpu

print(f"CODE_DIR: {CODE_DIR}")

CODE_DIR: /home/dw/github/decision-models-finetuning/code


In [2]:
# Load all included Wu 2025 survey rows and filter to paired electricity-plan conjoint tasks (`plan_choice`)
t_load0 = time.perf_counter()
wu2025_df = load_wu2025_df()
plan_choice_df = wu2025_df[wu2025_df["v"] == "plan_choice"].reset_index(drop=True)
load_wu_sec = time.perf_counter() - t_load0
if len(plan_choice_df) == 0:
    raise ValueError("No plan_choice rows found in wu2025_df")

# Summarize dataset size and load time: each answered task contributes 2 option rows to OLS
display(
    pd.Series(
        {
            "wu2025_total_rows": len(wu2025_df),
            "wu2025_questions": wu2025_df["v"].nunique(),
            "plan_choice_tasks": len(plan_choice_df),
            "plan_choice_option_rows": len(plan_choice_df) * 2,
            "respondents_with_plan_choice": plan_choice_df["CASEID"].nunique(),
            "distinct_displayed_pairs": plan_choice_df["qid"].nunique(),
            "load_sec": round(load_wu_sec, 4),
        },
        name="wu2025_overview",
    ).to_frame()
)

,wu2025_overview
wu2025_total_rows,90979.0000
wu2025_questions,46.0000
plan_choice_tasks,18591.0000
plan_choice_option_rows,37182.0000
respondents_with_plan_choice,1861.0000
distinct_displayed_pairs,8212.0000
load_sec,0.2054


In [3]:
# Ordered factor levels per attribute; index 0 in each list is the OLS reference level ($105, 10%, No)
WU2025_LEVELS = {
    ATTR_MONEY: ["$105", "$110", "$115", "$120", "$125"],
    ATTR_PERCENT: ["10%", "30%", "50%", "100%"],
    ATTR_LOCAL: ["No", "Yes"],
}

# Ordered names for the 8 non-intercept AMCE coefficients matching columns 1..8 of X
WU2025_BETA_NAMES = [
    *(f"{ATTR_MONEY}: {lvl} vs $105" for lvl in WU2025_LEVELS[ATTR_MONEY][1:]),
    *(f"{ATTR_PERCENT}: {lvl} vs 10%" for lvl in WU2025_LEVELS[ATTR_PERCENT][1:]),
    *(f"{ATTR_LOCAL}: {lvl} vs No" for lvl in WU2025_LEVELS[ATTR_LOCAL][1:]),
]

# SVD relative singular-value cutoff used to verify full column rank before inverting X
OLS_SVD_RCOND = 1e-8


def parse_wu2025_plan_description(desc: str) -> dict[str, str]:
    # Attribute display order is randomized per respondent, so split into 3 `Attribute: Level` clauses
    parts = desc.split("; ")
    if len(parts) != 3:
        raise ValueError(f"Expected 3 attribute clauses, got {desc!r}")
    kv: dict[str, str] = {}
    for part in parts:
        attr, sep, level = part.partition(": ")
        if not sep:
            raise ValueError(f"Malformed attribute clause {part!r} in {desc!r}")
        # Fail fast if any attribute name or level deviates from the known experimental design
        if attr not in WU2025_LEVELS:
            raise KeyError(f"Unexpected attribute {attr!r} in {desc!r}")
        if level not in WU2025_LEVELS[attr]:
            raise ValueError(f"Unexpected level {level!r} for {attr!r}")
        kv[attr] = level
    if set(kv.keys()) != set(WU2025_LEVELS.keys()):
        raise ValueError(f"Missing attributes in {desc!r}: got {sorted(kv.keys())}")
    return kv


def encode_wu2025_x_row(desc: str) -> list[float]:
    # Convert one option's description into [1.0 (intercept), 4 bill dummies, 3 renewables-% dummies, 1 local dummy]
    parsed = parse_wu2025_plan_description(desc)
    return [
        1.0,  # col 0: constant intercept (absorbs baseline choice level, dropped after projection)
        *[1.0 if parsed[ATTR_MONEY] == lvl else 0.0 for lvl in WU2025_LEVELS[ATTR_MONEY][1:]],      # cols 1..4: $110, $115, $120, $125
        *[1.0 if parsed[ATTR_PERCENT] == lvl else 0.0 for lvl in WU2025_LEVELS[ATTR_PERCENT][1:]],  # cols 5..7: 30%, 50%, 100%
        *[1.0 if parsed[ATTR_LOCAL] == lvl else 0.0 for lvl in WU2025_LEVELS[ATTR_LOCAL][1:]],      # col 8: Yes
    ]


def build_ols_projector(X: torch.Tensor) -> torch.Tensor:
    # Build fixed linear OLS operator A of shape (K, N) mapping N option outcomes -> K non-intercept betas
    if X.ndim != 2 or X.shape[0] < X.shape[1] or X.shape[1] < 2:
        raise ValueError(f"Expected 2D full-column design matrix X with N >= P >= 2, got shape {tuple(X.shape)}")
    X64 = X.to(dtype=torch.float64)
    # Fail fast via SVD if X is rank-deficient
    svals = torch.linalg.svdvals(X64)
    if float(svals[-1]) <= OLS_SVD_RCOND * float(svals[0]):
        raise ValueError(f"Rank-deficient design matrix X: min_sval={float(svals[-1]):.3e}, max_sval={float(svals[0]):.3e}")
    # pinv(X64) computes (X^T X)^{-1} X^T via SVD without squaring the condition number; [1:] drops the intercept
    return torch.linalg.pinv(X64)[1:]


def fit_ols_betas(A: torch.Tensor, outcome: torch.Tensor) -> torch.Tensor:
    # Linear OLS projection: beta = A @ outcome (differentiable whenever `outcome` comes from model predictions)
    if outcome.ndim != 1 or outcome.shape[0] != A.shape[1]:
        raise ValueError(f"Expected 1D outcome of length {A.shape[1]}, got shape {tuple(outcome.shape)}")
    return A @ outcome.to(dtype=A.dtype)


print(f"Configured {len(WU2025_BETA_NAMES)} non-intercept OLS beta terms (+ 1 intercept = {len(WU2025_BETA_NAMES) + 1} design columns)")

Configured 8 non-intercept OLS beta terms (+ 1 intercept = 9 design columns)


In [4]:
def compute_wu2025_loss(
    task_logits: torch.Tensor,
    y_flat: torch.Tensor,
    A: torch.Tensor,
    beta_lambda: float,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    # Validate paired 2-option task logits against flattened sample option targets and sample OLS projector A
    if task_logits.ndim != 2 or task_logits.shape[1] != 2:
        raise ValueError(f"Expected (num_tasks, 2) task_logits, got shape {tuple(task_logits.shape)}")
    num_rows = task_logits.numel()
    if y_flat.shape != (num_rows,) or A.shape[1] != num_rows:
        raise ValueError(f"Shape mismatch: num_rows={num_rows}, y_flat={tuple(y_flat.shape)}, A={tuple(A.shape)}")
    if y_flat.requires_grad or A.requires_grad:
        raise ValueError("Human choice targets `y_flat` and OLS projector `A` must be fixed (requires_grad=False)")

    # 1. Per-response KL divergence KL(y || p) (= CE on one-hot choices) averaged across all tasks in the respondent sample
    log_probs = torch.log_softmax(task_logits, dim=-1)
    y_2col = y_flat.view(-1, 2)
    kl_loss = torch.nn.functional.kl_div(log_probs, y_2col, reduction="batchmean")

    # 2. When beta_lambda == 0, compute sample OLS beta MSE under no_grad for logging so backward() stays pure KL/CE
    if beta_lambda == 0.0:
        with torch.no_grad():
            pred_probs_detached = torch.softmax(task_logits.detach(), dim=-1).reshape(-1)
            beta_loss = (fit_ols_betas(A, pred_probs_detached) - fit_ols_betas(A, y_flat)).pow(2).mean()
        return kl_loss, kl_loss, beta_loss

    # 3. Differentiable sample-level OLS beta MSE loss: mean((A_sample @ pred_probs - A_sample @ y_sample)^2)
    pred_probs = torch.softmax(task_logits, dim=-1).reshape(-1)
    pred_betas = fit_ols_betas(A, pred_probs)
    human_betas = fit_ols_betas(A, y_flat)
    beta_loss = (pred_betas - human_betas).pow(2).mean()

    # 4. Combined single-pass objective over the respondent sample: choice KL loss + beta_lambda * sample OLS beta MSE
    total_loss = kl_loss + beta_lambda * beta_loss
    return total_loss, kl_loss, beta_loss


print("Defined compute_wu2025_loss(task_logits, y_flat, A, beta_lambda)")

Defined compute_wu2025_loss(task_logits, y_flat, A, beta_lambda)


In [5]:
# Require CUDA and release any previously loaded `kev_model` before allocating a fresh 800M backbone
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required to run jaredpalmer/kev-0.8b in bf16 train mode")
if "kev_model" in globals():
    globals().pop("kev_model")
device = "cuda"
free_gpu()
torch.manual_seed(42)

# Build `jaredpalmer/kev-0.8b` with the v5 training configuration (bf16 frozen backbone + rank-64 LoRA + pointer head)
t_model0 = time.perf_counter()
kev_model = KevPointerModel.build(
    "jaredpalmer/kev-0.8b",
    device=device,
    lora_rank=64,
    weights_dtype="bf16",
)
kev_model.net.head.temperature = 1.0  # training uses raw uncalibrated temperature = 1.0
kev_model.train()                     # enable training mode (LoRA dropout active, trainable params require_grad)
torch.cuda.synchronize()
model_load_sec = time.perf_counter() - t_model0

# Summarize loaded model state, parameter counts, and GPU load time
trainable_params_count = sum(p.numel() for p in kev_model.trainable_parameters())
total_params_count = sum(p.numel() for p in kev_model.parameters())
display(
    pd.Series(
        {
            "base_model": "jaredpalmer/kev-0.8b",
            "mode": kev_model.mode,
            "training": kev_model.training,
            "weights_dtype": kev_model.meta.weights_dtype,
            "lora_rank": kev_model.meta.lora,
            "trainable_params": f"{trainable_params_count:,}",
            "total_params": f"{total_params_count:,}",
            "model_load_sec": round(model_load_sec, 3),
        },
        name="kev_800m_train_mode",
    ).to_frame()
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

,kev_800m_train_mode
base_model,jaredpalmer/kev-0.8b
mode,lora_rank 64
training,True
weights_dtype,bf16
lora_rank,64
trainable_params,"43,815,424"
total_params,"796,208,448"
model_load_sec,1.673


In [6]:
KRINER_AMCE_TERMS

[('C(airline_n)[T.2]', 'Airline: Frontier'),
 ('C(airline_n)[T.3]', 'Airline: Southwest'),
 ('C(airline_n)[T.4]', 'Airline: United'),
 ('C(fuel_type_n)[T.2]', 'Fuel: Sustainable aviation fuel'),
 ('C(baggage_n)[T.2]', 'Baggage: Checked bag+carry-on'),
 ('C(baggage_n)[T.3]', 'Baggage: Personal item only'),
 ('C(legroom_n)[T.2]', 'Space: Extra legroom'),
 ('C(seat_n)[T.2]', 'Seat: Middle'),
 ('C(seat_n)[T.3]', 'Seat: Window'),
 ('C(price_n)[T.2]', 'Price: $275'),
 ('C(price_n)[T.3]', 'Price: $300'),
 ('C(price_n)[T.4]', 'Price: $325')]

In [7]:
# Parse (column_name, non_reference_code, term_label) directly from Kriner 2026 Table 2 `AMCE_TERMS`
KRINER_TERM_SPECS: list[tuple[str, int, str]] = []
for patsy_term, term_label in KRINER_AMCE_TERMS:
    match = re.fullmatch(r"C\((\w+)\)\[T\.(\d+)\]", patsy_term)
    if match is None:
        raise ValueError(f"Unexpected Kriner2026 AMCE term syntax: {patsy_term!r}")
    KRINER_TERM_SPECS.append((match.group(1), int(match.group(2)), term_label))
KRINER_BETA_NAMES = [label for _, _, label in KRINER_TERM_SPECS]
KRINER_EXPECTED_OPTIONS = [text for _, text in KRINER_CHOICE_OPTIONS]


def encode_kriner2026_x_row(desc: str) -> list[float]:
    # Convert one displayed flight description into [1.0 (intercept), 12 Table-2 attribute-level indicators]
    codes = parse_flight_description(desc)
    return [1.0, *[1.0 if codes[col] == level_code else 0.0 for col, level_code, _ in KRINER_TERM_SPECS]]


# Build held-out cross-experiment evaluation bundle across all 5,000 Kriner 2026 `flight_choice` tasks (1,000 respondents)
t_kriner0 = time.perf_counter()
kriner2026_df = load_kriner2026_df()
kriner_flight_df = kriner2026_df[kriner2026_df["v"] == "flight_choice"].reset_index(drop=True)
if len(kriner_flight_df) == 0:
    raise ValueError("No flight_choice rows found in kriner2026_df")

kriner2026_records = build_long_records(
    kriner_flight_df,
    persona_cols=KRINER2026_PERSONA_COLS,
    include_country=False,
    year="kriner2026",
)
kriner2026_eval_bundle = group_records_by_state(
    kriner2026_records,
    view_fn=make_view,
    name="kriner2026_flight_choice",
    valid=True,
)

# Build fixed 12 x 10,000 OLS projector `A_kriner_pop` in exact `kriner2026_eval_bundle["records"]` order
kriner_x_rows: list[list[float]] = []
kriner_y_vals: list[float] = []
for task_idx, rec in enumerate(kriner2026_eval_bundle["records"]):
    if list(rec["option_texts"]) != KRINER_EXPECTED_OPTIONS:
        raise ValueError(f"Kriner task {task_idx}: expected {KRINER_EXPECTED_OPTIONS}, got {rec['option_texts']}")
    descs = rec["option_descriptions"]
    h_dist = [float(v) for v in rec["human_dist"]]
    if descs is None or len(descs) != 2 or len(h_dist) != 2 or sorted(h_dist) != [0.0, 1.0]:
        raise ValueError(f"Kriner task {task_idx}: invalid binary forced-choice record")
    for desc, y_val in zip(descs, h_dist, strict=True):
        kriner_x_rows.append(encode_kriner2026_x_row(desc))
        kriner_y_vals.append(y_val)

X_kriner_pop = torch.tensor(kriner_x_rows, dtype=torch.float64)
y_kriner_pop = torch.tensor(kriner_y_vals, dtype=torch.float64)
A_kriner_pop = build_ols_projector(X_kriner_pop)
kriner_human_betas = fit_ols_betas(A_kriner_pop, y_kriner_pop)
if bool((kriner_human_betas == 0).any()):
    raise ValueError("Zero Kriner2026 human AMCE encountered; MAPE denominator would be undefined")

# Verify machine-precision parity against `data.Kriner2026.evaluate_kriner2026_ols` (Table 2, col. 1)
_, _ref_kriner_df = evaluate_kriner2026_ols(
    kriner2026_eval_bundle,
    [r["human_dist"] for r in kriner2026_eval_bundle["records"]],
)
_max_table2_diff = float(
    (kriner_human_betas - torch.tensor(_ref_kriner_df["human"].to_numpy(), dtype=torch.float64)).abs().max()
)
if _max_table2_diff > 1e-12:
    raise AssertionError(f"PyTorch Kriner2026 OLS differs from evaluate_kriner2026_ols by {_max_table2_diff:.3e}")
kriner_bundle_prep_sec = time.perf_counter() - t_kriner0

print(
    f"Kriner2026 eval bundle: {len(kriner2026_eval_bundle['records']):,} tasks | "
    f"{len(kriner2026_eval_bundle['packed_views']):,} packed persona states | "
    f"A_kriner_pop shape: {tuple(A_kriner_pop.shape)} | max_table2_diff: {_max_table2_diff:.2e} | "
    f"prep_sec: {kriner_bundle_prep_sec:.3f}s"
)
display(pd.Series(kriner_human_betas.numpy(), index=KRINER_BETA_NAMES, name="kriner_human_pop_beta").round(4).to_frame())

Kriner2026 eval bundle: 5,000 tasks | 990 packed persona states | A_kriner_pop shape: (12, 10000) | max_table2_diff: 1.43e-15 | prep_sec: 0.297s


,kriner_human_pop_beta
Airline: Frontier,-0.0854
Airline: Southwest,-0.0238
Airline: United,-0.0220
Fuel: Sustainable aviation fuel,0.0691
Baggage: Checked bag+carry-on,0.0561
Baggage: Personal item only,-0.1393
Space: Extra legroom,0.0653
Seat: Middle,-0.0906
Seat: Window,0.0384
Price: $275,-0.0853


In [8]:
@torch.no_grad()
def validate_kriner2026_population_ols(
    model: KevPointerModel,
    eval_bundle: dict,
    A_eval: torch.Tensor,
    y_eval: torch.Tensor,
    human_eval_betas: torch.Tensor,
    beta_names: list[str],
) -> tuple[dict[str, float], pd.DataFrame]:
    # Time full evaluation wall-clock (including LoRA fusion / CudaGraphs setup + GPU inference + OLS projection)
    torch.cuda.synchronize()
    t_eval_wall0 = time.perf_counter()

    # Switch to eval mode and fuse current LoRA deltas + CudaGraphs for shared-state inference on Kriner 2026
    was_training = model.training
    model.eval()
    free_gpu()
    try:
        with torch.autocast("cuda", dtype=torch.bfloat16), fused_eval_context(model, eval_dtype=torch.bfloat16):
            preds, stats = predict_probs_grouped(model, eval_bundle, show_progress=True, return_stats=True)
    finally:
        # Always restore training mode and release temporary graph buffers so Wu 2025 training can resume immediately
        if was_training:
            model.train()
        free_gpu()

    # Fail fast if fused_eval_context failed to restore trainable parameter state or training mode
    if model.training != was_training:
        raise AssertionError(f"Expected model.training={was_training} after validation, got {model.training}")
    if any((not p.requires_grad) or p.numel() == 0 for p in model.trainable_parameters()):
        raise AssertionError("Trainable parameters were not cleanly restored after fused_eval_context")

    # Time CPU tensor stacking, Kriner 2026 choice KL computation, and `A_kriner_pop @ pred_probs` Table-2 AMCE projection
    t_ols0 = time.perf_counter()
    num_tasks = y_eval.numel() // 2
    if len(preds) != num_tasks:
        raise ValueError(f"Expected {num_tasks} Kriner2026 task predictions, got {len(preds)}")
    pred_probs_2col = torch.from_numpy(np.stack(preds, axis=0)).to(dtype=torch.float64)
    if pred_probs_2col.shape != (num_tasks, 2):
        raise ValueError(f"Expected ({num_tasks}, 2) predicted probabilities, got {tuple(pred_probs_2col.shape)}")

    kriner_kl_loss = float(
        torch.nn.functional.kl_div(
            pred_probs_2col.clamp_min(1e-12).log(),
            y_eval.view(-1, 2),
            reduction="batchmean",
        )
    )
    model_eval_betas = fit_ols_betas(A_eval, pred_probs_2col.reshape(-1))
    beta_diff = model_eval_betas - human_eval_betas
    eval_ols_ms = (time.perf_counter() - t_ols0) * 1000.0
    eval_wall_sec = time.perf_counter() - t_eval_wall0

    # Report out-of-experiment Kriner 2026 OLS / KL metrics side-by-side with detailed evaluation timing metrics
    metrics = {
        "kriner_kl_loss": kriner_kl_loss,
        "kriner_beta_mse": float(beta_diff.pow(2).mean()),
        "kriner_beta_mae": float(beta_diff.abs().mean()),
        "kriner_beta_mape_pct": float((beta_diff.abs() / human_eval_betas.abs()).mean() * 100.0),
        "spearman_rho": float(spearmanr(model_eval_betas.numpy(), human_eval_betas.numpy()).statistic),
        "eval_prep_sec": float(stats["prep_sec"]),
        "eval_gpu_sec": float(stats["gpu_sec"]),
        "eval_ols_ms": eval_ols_ms,
        "eval_wall_sec": eval_wall_sec,
        "eval_ms_per_task": (eval_wall_sec * 1000.0) / num_tasks,
    }
    beta_df = pd.DataFrame(
        {
            "human_pop_beta": human_eval_betas.numpy(),
            "model_pop_beta": model_eval_betas.numpy(),
            "diff": beta_diff.numpy(),
            "abs_diff": beta_diff.abs().numpy(),
        },
        index=pd.Index(beta_names, name="kriner_term"),
    )
    return metrics, beta_df


# Run Step 0 out-of-experiment baseline validation on Kriner 2026 (`flight_choice`) before any Wu 2025 training
step0_val_metrics, step0_beta_df = validate_kriner2026_population_ols(
    kev_model,
    kriner2026_eval_bundle,
    A_kriner_pop,
    y_kriner_pop,
    kriner_human_betas,
    KRINER_BETA_NAMES,
)
display(pd.Series(step0_val_metrics, name="step_0_kriner2026_val").round(4).to_frame())
display(step0_beta_df.round(4))

Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

,step_0_kriner2026_val
kriner_kl_loss,1.0849
kriner_beta_mse,0.0178
kriner_beta_mae,0.1143
kriner_beta_mape_pct,133.5286
spearman_rho,-0.2657
eval_prep_sec,0.0435
eval_gpu_sec,8.8700
eval_ols_ms,1.6067
eval_wall_sec,10.5986
eval_ms_per_task,2.1197


,human_pop_beta,model_pop_beta,diff,abs_diff
kriner_term,,,,
Airline: Frontier,-0.0854,0.0344,0.1198,0.1198
Airline: Southwest,-0.0238,0.0267,0.0505,0.0505
Airline: United,-0.0220,0.0010,0.0230,0.0230
Fuel: Sustainable aviation fuel,0.0691,-0.0574,-0.1265,0.1265
Baggage: Checked bag+carry-on,0.0561,-0.0435,-0.0996,0.0996
Baggage: Personal item only,-0.1393,-0.0030,0.1362,0.1362
Space: Extra legroom,0.0653,-0.0813,-0.1466,0.1466
Seat: Middle,-0.0906,-0.0057,0.0849,0.0849
Seat: Window,0.0384,0.0086,-0.0299,0.0299


In [9]:
# Snapshot all 376 untrained LoRA + pointer-head tensors on CPU so every `(sample_size, beta_lambda)` run starts identically
INIT_TRAINABLE_STATE = {
    name: param.detach().cpu().clone()
    for name, param in kev_model.named_parameters()
    if param.requires_grad
}


def reset_kev_trainable_weights(model: KevPointerModel, init_state: dict[str, torch.Tensor]) -> None:
    # Fail fast if the model's trainable parameter set changed before copying Step-0 weights back onto GPU
    current_names = {name for name, param in model.named_parameters() if param.requires_grad}
    if current_names != set(init_state.keys()):
        raise ValueError("Trainable parameter names do not match INIT_TRAINABLE_STATE snapshot")
    with torch.no_grad():
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.copy_(init_state[name].to(device=param.device, dtype=param.dtype))
                param.grad = None
    model.train()
    free_gpu()


# Pre-tokenize all 1,861 respondents once, storing `enc`, `X_resp` (CPU float64), and `y_resp` (GPU float64)
# so any respondent `sample_size` can stack `X_sample` and `y_sample` on the fly without re-tokenizing
t_resp_prep0 = time.perf_counter()
wu2025_respondent_items: list[dict] = []
single_resp_full_rank_count = 0

for r_caseid, r_group in tqdm(
    plan_choice_df.groupby("CASEID", sort=False),
    total=plan_choice_df["CASEID"].nunique(),
    desc="Preparing respondents",
):
    r_df = r_group.reset_index(drop=True)
    x_rows: list[list[float]] = []
    y_vals: list[float] = []
    for t_idx, row in enumerate(r_df.itertuples(index=False)):
        # Validate every respondent task's options, descriptions, and binary one-hot choice vector
        if list(row.option_texts) != CONJOINT_OPTIONS:
            raise ValueError(f"{r_caseid} task {t_idx}: expected {CONJOINT_OPTIONS}, got {row.option_texts}")
        if row.option_descriptions is None or len(row.option_descriptions) != 2:
            raise ValueError(f"{r_caseid} task {t_idx}: expected 2 option_descriptions")
        one_hot = [float(v) for v in row.one_hot]
        if len(one_hot) != 2 or sorted(one_hot) != [0.0, 1.0]:
            raise ValueError(f"{r_caseid} task {t_idx}: invalid binary one_hot {one_hot}")
        for desc, y_val in zip(row.option_descriptions, one_hot, strict=True):
            x_rows.append(encode_wu2025_x_row(desc))
            y_vals.append(y_val)

    # Tokenize the respondent's shared persona state + choice tasks once and verify target alignment
    r_records = build_long_records(r_df, persona_cols=WU2025_PERSONA_COLS, include_country=False, year="wu2025")
    r_enc, r_targets = kev_model.prepare(make_grouped_view(r_records))
    r_X = torch.tensor(x_rows, dtype=torch.float64)
    r_y = torch.tensor(y_vals, dtype=torch.float64, device=device)
    prepared_2col = torch.stack([q["target"].to(dtype=torch.float64, device=device) for q in r_targets], dim=0)
    if not torch.equal(prepared_2col, r_y.view(-1, 2)):
        raise AssertionError(f"{r_caseid}: Kev prepared target order does not match y_resp")

    if r_X.shape[0] >= r_X.shape[1]:
        r_svals = torch.linalg.svdvals(r_X)
        if float(r_svals[-1]) > OLS_SVD_RCOND * float(r_svals[0]):
            single_resp_full_rank_count += 1

    wu2025_respondent_items.append(
        {
            "caseid": str(r_caseid),
            "num_tasks": len(r_df),
            "enc": r_enc,
            "X_resp": r_X,
            "y_resp": r_y,
        }
    )

resp_prep_sec = time.perf_counter() - t_resp_prep0
if (len(wu2025_respondent_items), single_resp_full_rank_count) != (1861, 1725):
    raise AssertionError(
        f"Unexpected respondent counts: {(len(wu2025_respondent_items), single_resp_full_rank_count)}"
    )

# Registry keyed by `(sample_size, beta_lambda)` storing full step/timing logs and population OLS trajectories
experiment_runs: dict[tuple[int, float], dict] = {}

display(
    pd.Series(
        {
            "total_respondents": len(wu2025_respondent_items),
            "single_respondent_full_rank": single_resp_full_rank_count,
            "single_respondent_rank_deficient": len(wu2025_respondent_items) - single_resp_full_rank_count,
            "snapshot_trainable_tensors": len(INIT_TRAINABLE_STATE),
            "respondent_prep_sec": round(resp_prep_sec, 3),
            "ms_per_respondent_prep": round((resp_prep_sec * 1000.0) / len(wu2025_respondent_items), 3),
        },
        name="epoch_prep_summary",
    ).to_frame()
)

Preparing respondents:   0%|          | 0/1861 [00:00<?, ?it/s]

,epoch_prep_summary
total_respondents,1861.000
single_respondent_full_rank,1725.000
single_respondent_rank_deficient,136.000
snapshot_trainable_tensors,376.000
respondent_prep_sec,7.183
ms_per_respondent_prep,3.860


In [10]:
# Shared epoch hyperparameters; each optimization step draws a sample of `sample_size` human respondents
EPOCH_SEED = 42
BASE_LR_SAMPLE_1 = 2e-5               # validated 1-respondent/step learning rate from v1
EPOCH_WEIGHT_DECAY = 0.01
NUM_EPOCH_VAL_CHECKS = 4              # validate population OLS ~4 times per epoch (~every quarter of respondents)
FWD_CHUNK_RESPONDENTS = 8             # max respondents batched into one `shared_prefix=True` GPU kernel call
MAX_UNCHECKPOINTED_SAMPLE_SIZE = 8    # enable per-layer activation checkpointing when `sample_size > 8` to bound VRAM


def scale_lr_for_sample_size(sample_size: int) -> float:
    # Square-root AdamW LR scaling keeps 1-epoch parameter travel invariant as `steps_per_epoch` shrinks by 1/sample_size
    if not isinstance(sample_size, int) or isinstance(sample_size, bool) or sample_size < 1:
        raise ValueError(f"sample_size must be a positive integer, got {sample_size!r}")
    return BASE_LR_SAMPLE_1 * math.sqrt(sample_size)


def run_wu2025_epoch(
    sample_size: int,
    beta_lambda: float,
    lr: float,
    num_epochs: int,
) -> dict:
    # Fail fast on invalid sample_size, num_epochs, beta_lambda, or learning rate
    if not isinstance(sample_size, int) or isinstance(sample_size, bool) or sample_size < 1:
        raise ValueError(f"sample_size must be a positive integer, got {sample_size!r}")
    if not isinstance(num_epochs, int) or isinstance(num_epochs, bool) or num_epochs < 1:
        raise ValueError(f"num_epochs must be a positive integer, got {num_epochs!r}")
    if beta_lambda < 0.0:
        raise ValueError(f"beta_lambda must be non-negative, got {beta_lambda}")
    if lr <= 0.0:
        raise ValueError(f"lr must be positive, got {lr}")

    t_wall0 = time.perf_counter()

    # Reset weights to exact Step-0 initialization and re-seed RNGs so respondent sample order matches across runs
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    reset_kev_trainable_weights(kev_model, INIT_TRAINABLE_STATE)
    if sample_size > MAX_UNCHECKPOINTED_SAMPLE_SIZE:
        kev_model.enable_checkpointing()
    else:
        kev_model.net.lm.gradient_checkpointing_disable()

    rng = random.Random(EPOCH_SEED)
    torch.manual_seed(EPOCH_SEED)
    torch.cuda.manual_seed_all(EPOCH_SEED)

    # One epoch = one pass over all respondents; respondents are reshuffled at the start of every epoch
    base_items = list(wu2025_respondent_items)
    total_respondents = len(base_items)
    steps_per_epoch = math.ceil(total_respondents / sample_size)
    total_steps = steps_per_epoch * num_epochs
    val_every = max(1, steps_per_epoch // NUM_EPOCH_VAL_CHECKS)
    postfix_every = max(1, steps_per_epoch // 60)
    rolling_steps = max(1, round(300 / sample_size))

    # Configure AdamW + one OneCycleLR schedule spanning all `num_epochs * steps_per_epoch` steps
    trainable_params = list(kev_model.trainable_parameters())
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=EPOCH_WEIGHT_DECAY, fused=True)
    if total_steps > 1:
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=lr,
            total_steps=total_steps,
            pct_start=0.1,
            div_factor=25.0,
            final_div_factor=1e4,
        )
    else:
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda _: 1.0)

    step_rows: list[dict] = []
    val_rows: list[dict] = [
        {"step": 0, "epoch": 0, "respondents_seen": 0, "train_elapsed_sec": 0.0, **step0_val_metrics}
    ]
    final_beta_df = step0_beta_df
    cum_train_sec = 0.0
    cum_eval_sec = 0.0
    respondents_seen = 0

    pbar = tqdm(range(1, total_steps + 1), total=total_steps, desc=f"S={sample_size} | lambda={beta_lambda:g}")
    for step_idx in pbar:
        epoch_idx = (step_idx - 1) // steps_per_epoch + 1
        step_in_epoch = (step_idx - 1) % steps_per_epoch + 1
        if step_in_epoch == 1:
            epoch_items = list(base_items)
            rng.shuffle(epoch_items)
        start_idx = (step_in_epoch - 1) * sample_size
        sample_items = epoch_items[start_idx : start_idx + sample_size]
        n_sample_resp = len(sample_items)
        respondents_seen += n_sample_resp

        torch.cuda.synchronize()
        t_step0 = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)

        # 1. Single shared-prefix forward pass across all respondents in this sample (batched up to FWD_CHUNK_RESPONDENTS)
        sample_logit_tensors: list[torch.Tensor] = []
        with torch.autocast("cuda", dtype=torch.bfloat16):
            for c_start in range(0, n_sample_resp, FWD_CHUNK_RESPONDENTS):
                chunk_encs = [it["enc"] for it in sample_items[c_start : c_start + FWD_CHUNK_RESPONDENTS]]
                chunk_logits_lists = kev_model.forward(chunk_encs, shared_prefix=True)
                for resp_logits_list in chunk_logits_lists:
                    sample_logit_tensors.extend(z.to(dtype=torch.float64) for z in resp_logits_list)
        task_logits = torch.stack(sample_logit_tensors, dim=0)
        torch.cuda.synchronize()
        t_after_fwd = time.perf_counter()

        # 2. Stack `X_sample` and `y_sample` over all respondents in this step and fit sample-level OLS if rank == 9/9
        X_sample = torch.cat([it["X_resp"] for it in sample_items], dim=0)
        y_sample = torch.cat([it["y_resp"] for it in sample_items], dim=0)
        if X_sample.shape[0] >= X_sample.shape[1]:
            svals = torch.linalg.svdvals(X_sample)
            has_ols = float(svals[-1]) > OLS_SVD_RCOND * float(svals[0])
        else:
            has_ols = False

        if has_ols:
            A_sample = torch.linalg.pinv(X_sample)[1:].to(device=device)
            total_loss, kl_loss, beta_loss = compute_wu2025_loss(
                task_logits,
                y_sample,
                A_sample,
                beta_lambda=beta_lambda,
            )
            raw_beta_val = float(beta_loss.detach().cpu())
        else:
            A_sample = None
            beta_loss = None
            log_probs = torch.log_softmax(task_logits, dim=-1)
            kl_loss = torch.nn.functional.kl_div(log_probs, y_sample.view(-1, 2), reduction="batchmean")
            total_loss = kl_loss
            raw_beta_val = float("nan")

        kl_val = float(kl_loss.detach().cpu())
        total_val = float(total_loss.detach().cpu())
        torch.cuda.synchronize()
        t_after_loss = time.perf_counter()

        # 3. Single backward pass across the respondent sample, gradient clipping, AdamW update, and LR schedule step
        total_loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        if not bool(torch.isfinite(grad_norm)):
            raise FloatingPointError(f"Non-finite gradient norm {float(grad_norm)} at step {step_idx}")
        grad_norm_val = float(grad_norm.detach().cpu())
        optimizer.step()
        if step_idx < total_steps:
            scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        # Release per-step autograd references immediately so no computation graph lingers across steps or evaluations
        del sample_logit_tensors, task_logits, X_sample, y_sample, A_sample, total_loss, kl_loss, beta_loss
        torch.cuda.synchronize()
        t_after_bwd = time.perf_counter()

        fwd_ms = (t_after_fwd - t_step0) * 1000.0
        loss_ms = (t_after_loss - t_after_fwd) * 1000.0
        bwd_opt_ms = (t_after_bwd - t_after_loss) * 1000.0
        step_ms = (t_after_bwd - t_step0) * 1000.0
        resp_ms = step_ms / n_sample_resp
        cum_train_sec += t_after_bwd - t_step0

        step_rows.append(
            {
                "step": step_idx,
                "epoch": epoch_idx,
                "num_respondents": n_sample_resp,
                "respondents_seen": respondents_seen,
                "has_ols": has_ols,
                "kl_loss": kl_val,
                "raw_beta_loss": raw_beta_val,
                "weighted_beta_loss": beta_lambda * raw_beta_val if has_ols else 0.0,
                "total_loss": total_val,
                "grad_norm": grad_norm_val,
                "fwd_ms": fwd_ms,
                "loss_ms": loss_ms,
                "bwd_opt_ms": bwd_opt_ms,
                "step_ms": step_ms,
                "ms_per_respondent": resp_ms,
            }
        )

        # Update live progress bar with rolling (~300 respondents) statistical losses and step / per-respondent timings
        if step_idx % postfix_every == 0 or step_idx == total_steps:
            recent = step_rows[-rolling_steps:]
            recent_betas = [r["raw_beta_loss"] for r in recent if r["has_ols"]]
            pbar.set_postfix(
                kl=f"{float(np.mean([r['kl_loss'] for r in recent])):.3f}",
                beta=f"{float(np.mean(recent_betas)):.4f}" if recent_betas else "n/a",
                step_ms=f"{float(np.mean([r['step_ms'] for r in recent])):.1f}",
                resp_ms=f"{float(np.mean([r['ms_per_respondent'] for r in recent])):.1f}",
            )

        # 4. Evaluate out-of-experiment Kriner 2026 population OLS at quarter-epoch checkpoints and every epoch end
        if step_in_epoch % val_every == 0 or step_in_epoch == steps_per_epoch:
            v_metrics, final_beta_df = validate_kriner2026_population_ols(
                kev_model,
                kriner2026_eval_bundle,
                A_kriner_pop,
                y_kriner_pop,
                kriner_human_betas,
                KRINER_BETA_NAMES,
            )
            cum_eval_sec += v_metrics["eval_wall_sec"]
            val_rows.append(
                {
                    "step": step_idx,
                    "epoch": epoch_idx,
                    "respondents_seen": respondents_seen,
                    "train_elapsed_sec": cum_train_sec,
                    **v_metrics,
                }
            )
            tqdm.write(
                f"[S={sample_size} | lambda={beta_lambda:g} | epoch {epoch_idx}/{num_epochs} | step {step_idx:4d}/{total_steps} ({respondents_seen}/{total_respondents * num_epochs} resp)] "
                f"kriner_kl={v_metrics['kriner_kl_loss']:.4f} | "
                f"kriner_beta_mse={v_metrics['kriner_beta_mse']:.5f} | "
                f"MAPE={v_metrics['kriner_beta_mape_pct']:.2f}% | "
                f"rho={v_metrics['spearman_rho']:.3f} | "
                f"train={cum_train_sec:.1f}s | "
                f"eval_gpu={v_metrics['eval_gpu_sec']:.2f}s | "
                f"eval_wall={v_metrics['eval_wall_sec']:.2f}s"
            )

    total_wall_sec = time.perf_counter() - t_wall0
    step_df = pd.DataFrame(step_rows)
    val_history_df = pd.DataFrame(val_rows).set_index("step")
    ols_steps = step_df[step_df["has_ols"]]
    edge_window = max(1, round(0.15 * len(step_df)))          # first/last ~15% of all training steps regardless of sample_size
    ols_edge_window = max(1, round(0.15 * len(ols_steps)))

    # Summarize both statistical loss metrics and per-step / per-respondent / eval / wall-clock timing metrics
    summary = pd.Series(
        {
            "sample_size": sample_size,
            "beta_lambda": beta_lambda,
            "lr": lr,
            "num_epochs": num_epochs,
            "steps_per_epoch": steps_per_epoch,
            "total_steps": total_steps,
            "rank_deficient_steps": int((~step_df["has_ols"]).sum()),
            "train_kl_first15pct": float(step_df["kl_loss"].iloc[:edge_window].mean()),
            "train_kl_last15pct": float(step_df["kl_loss"].iloc[-edge_window:].mean()),
            "train_raw_beta_first15pct": float(ols_steps["raw_beta_loss"].iloc[:ols_edge_window].mean()),
            "train_raw_beta_last15pct": float(ols_steps["raw_beta_loss"].iloc[-ols_edge_window:].mean()),
            "train_weighted_beta_last15pct": float(ols_steps["weighted_beta_loss"].iloc[-ols_edge_window:].mean()),
            "train_total_first15pct": float(step_df["total_loss"].iloc[:edge_window].mean()),
            "train_total_last15pct": float(step_df["total_loss"].iloc[-edge_window:].mean()),
            "mean_grad_norm": float(step_df["grad_norm"].mean()),
            "mean_fwd_ms": float(step_df["fwd_ms"].mean()),
            "mean_loss_ms": float(step_df["loss_ms"].mean()),
            "mean_bwd_opt_ms": float(step_df["bwd_opt_ms"].mean()),
            "mean_step_ms": float(step_df["step_ms"].mean()),
            "mean_ms_per_respondent": (cum_train_sec * 1000.0) / respondents_seen,
            "mean_eval_wall_sec": float(val_history_df["eval_wall_sec"].iloc[1:].mean()),
            "mean_eval_gpu_sec": float(val_history_df["eval_gpu_sec"].iloc[1:].mean()),
            "train_only_sec": cum_train_sec,
            "eval_only_sec": cum_eval_sec,
            "total_wall_sec": total_wall_sec,
        },
        name=f"S_{sample_size}_lambda_{beta_lambda:g}_summary",
    )
    display(summary.round(5).to_frame())
    display(val_history_df.round(5))

    return {
        "sample_size": sample_size,
        "beta_lambda": beta_lambda,
        "lr": lr,
        "num_epochs": num_epochs,
        "summary": summary,
        "step_df": step_df,
        "val_history_df": val_history_df,
        "final_beta_df": final_beta_df,
    }


print("Defined scale_lr_for_sample_size(sample_size) and run_wu2025_epoch(sample_size, beta_lambda, lr, num_epochs)")

Defined scale_lr_for_sample_size(sample_size) and run_wu2025_epoch(sample_size, beta_lambda, lr)


In [11]:
validate_kriner2026_population_ols(
                kev_model,
                kriner2026_eval_bundle,
                A_kriner_pop,
                y_kriner_pop,
                kriner_human_betas,
                KRINER_BETA_NAMES,
            )

Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

({'kriner_kl_loss': 1.0849053503715131,
  'kriner_beta_mse': 0.01775759228774546,
  'kriner_beta_mae': 0.11429308404320225,
  'kriner_beta_mape_pct': 133.52860442223414,
  'spearman_rho': -0.2657342657342658,
  'eval_prep_sec': 0.0036915430100634694,
  'eval_gpu_sec': 8.611311506014317,
  'eval_ols_ms': 1.82033097371459,
  'eval_wall_sec': 9.748525261995383,
  'eval_ms_per_task': 1.9497050523990764},
                                  human_pop_beta  model_pop_beta      diff  \
 kriner_term                                                                 
 Airline: Frontier                     -0.085437        0.034381  0.119818   
 Airline: Southwest                    -0.023794        0.026743  0.050538   
 Airline: United                       -0.021977        0.001039  0.023016   
 Fuel: Sustainable aviation fuel        0.069068       -0.057438 -0.126506   
 Baggage: Checked bag+carry-on          0.056054       -0.043539 -0.099594   
 Baggage: Personal item only           -0.139254  

In [12]:
# Number of human respondents sampled per optimization step (edit `SAMPLE_SIZE` here to run 8, 16, 32, etc.)
SAMPLE_SIZE = 16
# Number of full passes over all Wu 2025 respondents per run
NUM_EPOCHS = 3
SAMPLE_LR = scale_lr_for_sample_size(SAMPLE_SIZE)
steps_per_epoch = math.ceil(len(wu2025_respondent_items) / SAMPLE_SIZE)
total_wu_option_rows = sum(int(it["X_resp"].shape[0]) for it in wu2025_respondent_items)

display(
    pd.Series(
        {
            "sample_size": SAMPLE_SIZE,
            "num_epochs": NUM_EPOCHS,
            "steps_per_epoch": steps_per_epoch,
            "total_steps": steps_per_epoch * NUM_EPOCHS,
            "mean_option_rows_per_step": round(total_wu_option_rows / steps_per_epoch, 2),
            "scaled_max_lr": SAMPLE_LR,
            "gradient_checkpointing": SAMPLE_SIZE > MAX_UNCHECKPOINTED_SAMPLE_SIZE,
        },
        name="selected_sample_size_config",
    ).to_frame()
)

,selected_sample_size_config
sample_size,16
steps_per_epoch,117
mean_option_rows_per_step,317.79
scaled_max_lr,0.00008
gradient_checkpointing,True


In [13]:
# Run `NUM_EPOCHS` epochs at `SAMPLE_SIZE` respondents/step with `beta_lambda = 0.0` (pure per-response KL/CE baseline)
experiment_runs[(SAMPLE_SIZE, 0.0)] = run_wu2025_epoch(
    sample_size=SAMPLE_SIZE,
    beta_lambda=0.0,
    lr=SAMPLE_LR,
    num_epochs=NUM_EPOCHS,
)

S=16 | lambda=0:   0%|          | 0/117 [00:00<?, ?it/s]

Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

[S=16 | lambda=0 | step   29/117 (464/1861 resp)] kriner_kl=0.6452 | kriner_beta_mse=0.00747 | MAPE=84.87% | rho=0.923 | train=65.7s | eval_gpu=8.40s | eval_wall=9.44s


Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

[S=16 | lambda=0 | step   58/117 (928/1861 resp)] kriner_kl=0.6268 | kriner_beta_mse=0.00299 | MAPE=68.34% | rho=0.930 | train=130.3s | eval_gpu=7.85s | eval_wall=8.74s


Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

... (3 outputs left out)

Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

[S=16 | lambda=0 | step  117/117 (1861/1861 resp)] kriner_kl=0.6268 | kriner_beta_mse=0.00389 | MAPE=75.40% | rho=0.937 | train=250.2s | eval_gpu=7.84s | eval_wall=8.71s


,S_16_lambda_0_summary
sample_size,16.00000
beta_lambda,0.00000
lr,0.00008
steps_per_epoch,117.00000
rank_deficient_steps,0.00000
train_kl_first15pct,0.64176
train_kl_last15pct,0.54015
train_raw_beta_first15pct,0.01783
train_raw_beta_last15pct,0.01087
train_weighted_beta_last15pct,0.00000


,respondents_seen,train_elapsed_sec,kriner_kl_loss,kriner_beta_mse,kriner_beta_mae,kriner_beta_mape_pct,spearman_rho,eval_prep_sec,eval_gpu_sec,eval_ols_ms,eval_wall_sec,eval_ms_per_task
step,,,,,,,,,,,,
0,0,0.00000,1.08491,0.01776,0.11429,133.52860,-0.26573,0.04481,8.95083,1.80457,10.71794,2.14359
29,464,65.73115,0.64521,0.00747,0.07335,84.87320,0.92308,0.00363,8.40162,2.11243,9.43981,1.88796
58,928,130.33791,0.62678,0.00299,0.04305,68.34188,0.93007,0.00318,7.84674,2.02715,8.74360,1.74872
87,1392,189.93091,0.62727,0.00383,0.05474,75.17656,0.91608,0.00315,7.85804,2.05247,8.73144,1.74629
116,1856,249.60197,0.62686,0.00389,0.05537,75.37790,0.93706,0.00308,7.84101,2.05030,8.73188,1.74638
117,1861,250.21588,0.62682,0.00389,0.05537,75.39640,0.93706,0.00312,7.84478,1.94703,8.71331,1.74266


In [14]:
# Run `NUM_EPOCHS` epochs at `SAMPLE_SIZE` respondents/step with `beta_lambda = 1.0` (mild sample OLS beta weight)
experiment_runs[(SAMPLE_SIZE, 1.0)] = run_wu2025_epoch(
    sample_size=SAMPLE_SIZE,
    beta_lambda=1.0,
    lr=SAMPLE_LR,
    num_epochs=NUM_EPOCHS,
)

S=16 | lambda=1:   0%|          | 0/117 [00:00<?, ?it/s]

Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

[S=16 | lambda=1 | step   29/117 (464/1861 resp)] kriner_kl=0.6480 | kriner_beta_mse=0.00787 | MAPE=84.77% | rho=0.923 | train=59.7s | eval_gpu=7.84s | eval_wall=8.72s


Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

[S=16 | lambda=1 | step   58/117 (928/1861 resp)] kriner_kl=0.6235 | kriner_beta_mse=0.00318 | MAPE=66.22% | rho=0.902 | train=119.4s | eval_gpu=7.84s | eval_wall=8.72s


Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

... (3 outputs left out)

Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

[S=16 | lambda=1 | step  117/117 (1861/1861 resp)] kriner_kl=0.6238 | kriner_beta_mse=0.00294 | MAPE=63.41% | rho=0.895 | train=238.8s | eval_gpu=7.83s | eval_wall=8.70s


,S_16_lambda_1_summary
sample_size,16.00000
beta_lambda,1.00000
lr,0.00008
steps_per_epoch,117.00000
rank_deficient_steps,0.00000
train_kl_first15pct,0.63987
train_kl_last15pct,0.53745
train_raw_beta_first15pct,0.01654
train_raw_beta_last15pct,0.01020
train_weighted_beta_last15pct,0.01020


,respondents_seen,train_elapsed_sec,kriner_kl_loss,kriner_beta_mse,kriner_beta_mae,kriner_beta_mape_pct,spearman_rho,eval_prep_sec,eval_gpu_sec,eval_ols_ms,eval_wall_sec,eval_ms_per_task
step,,,,,,,,,,,,
0,0,0.00000,1.08491,0.01776,0.11429,133.52860,-0.26573,0.04481,8.95083,1.80457,10.71794,2.14359
29,464,59.67633,0.64805,0.00787,0.07451,84.76693,0.92308,0.00317,7.84365,2.04769,8.71828,1.74366
58,928,119.43534,0.62346,0.00318,0.04683,66.22400,0.90210,0.00313,7.84196,2.00667,8.72414,1.74483
87,1392,178.76321,0.62902,0.00299,0.04442,64.13981,0.90210,0.00320,7.83038,1.96866,8.71547,1.74309
116,1856,238.14888,0.62393,0.00294,0.04328,63.41421,0.89510,0.00315,7.82604,1.99048,8.70500,1.74100
117,1861,238.75339,0.62384,0.00294,0.04329,63.40753,0.89510,0.00315,7.82737,2.31197,8.70037,1.74007


In [15]:
# Run `NUM_EPOCHS` epochs at `SAMPLE_SIZE` respondents/step with `beta_lambda = 5.0` (moderate sample OLS beta weight)
experiment_runs[(SAMPLE_SIZE, 5.0)] = run_wu2025_epoch(
    sample_size=SAMPLE_SIZE,
    beta_lambda=5.0,
    lr=SAMPLE_LR,
    num_epochs=NUM_EPOCHS,
)

S=16 | lambda=5:   0%|          | 0/117 [00:00<?, ?it/s]

Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

[S=16 | lambda=5 | step   29/117 (464/1861 resp)] kriner_kl=0.6487 | kriner_beta_mse=0.00485 | MAPE=75.56% | rho=0.776 | train=59.4s | eval_gpu=7.82s | eval_wall=8.70s


Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

[S=16 | lambda=5 | step   58/117 (928/1861 resp)] kriner_kl=0.6318 | kriner_beta_mse=0.00389 | MAPE=69.03% | rho=0.832 | train=118.8s | eval_gpu=7.83s | eval_wall=8.72s


Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

... (3 outputs left out)

Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

[S=16 | lambda=5 | step  117/117 (1861/1861 resp)] kriner_kl=0.6316 | kriner_beta_mse=0.00357 | MAPE=67.46% | rho=0.860 | train=238.1s | eval_gpu=7.82s | eval_wall=8.67s


,S_16_lambda_5_summary
sample_size,16.00000
beta_lambda,5.00000
lr,0.00008
steps_per_epoch,117.00000
rank_deficient_steps,0.00000
train_kl_first15pct,0.64474
train_kl_last15pct,0.54127
train_raw_beta_first15pct,0.01618
train_raw_beta_last15pct,0.01045
train_weighted_beta_last15pct,0.05227


,respondents_seen,train_elapsed_sec,kriner_kl_loss,kriner_beta_mse,kriner_beta_mae,kriner_beta_mape_pct,spearman_rho,eval_prep_sec,eval_gpu_sec,eval_ols_ms,eval_wall_sec,eval_ms_per_task
step,,,,,,,,,,,,
0,0,0.00000,1.08491,0.01776,0.11429,133.52860,-0.26573,0.04481,8.95083,1.80457,10.71794,2.14359
29,464,59.36807,0.64865,0.00485,0.06092,75.55885,0.77622,0.00310,7.82418,2.07053,8.70357,1.74071
58,928,118.83911,0.63184,0.00389,0.05382,69.03357,0.83217,0.00309,7.83347,1.99585,8.71723,1.74345
87,1392,178.12647,0.63678,0.00401,0.05489,70.40777,0.86014,0.00318,7.82422,2.02880,8.69181,1.73836
116,1856,237.49583,0.63172,0.00357,0.05088,67.55508,0.86014,0.00310,7.82175,1.98792,8.68673,1.73735
117,1861,238.10123,0.63163,0.00357,0.05084,67.46192,0.86014,0.00304,7.82298,1.94715,8.66754,1.73351


In [16]:
# Run `NUM_EPOCHS` epochs at `SAMPLE_SIZE` respondents/step with `beta_lambda = 20.0` (strong sample OLS beta weight)
experiment_runs[(SAMPLE_SIZE, 20.0)] = run_wu2025_epoch(
    sample_size=SAMPLE_SIZE,
    beta_lambda=20.0,
    lr=SAMPLE_LR,
    num_epochs=NUM_EPOCHS,
)

S=16 | lambda=20:   0%|          | 0/117 [00:00<?, ?it/s]

Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

[S=16 | lambda=20 | step   29/117 (464/1861 resp)] kriner_kl=0.6409 | kriner_beta_mse=0.00334 | MAPE=69.16% | rho=0.825 | train=59.4s | eval_gpu=7.82s | eval_wall=8.68s


Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

[S=16 | lambda=20 | step   58/117 (928/1861 resp)] kriner_kl=0.6285 | kriner_beta_mse=0.00348 | MAPE=72.31% | rho=0.846 | train=118.8s | eval_gpu=7.81s | eval_wall=8.65s


Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

... (3 outputs left out)

Evaluating kriner2026_flight_choice (shared-state):   0%|          | 0/22 [00:00<?, ?batch/s]

[S=16 | lambda=20 | step  117/117 (1861/1861 resp)] kriner_kl=0.6313 | kriner_beta_mse=0.00329 | MAPE=66.31% | rho=0.839 | train=238.0s | eval_gpu=7.81s | eval_wall=8.63s


,S_16_lambda_20_summary
sample_size,16.00000
beta_lambda,20.00000
lr,0.00008
steps_per_epoch,117.00000
rank_deficient_steps,0.00000
train_kl_first15pct,0.64587
train_kl_last15pct,0.54544
train_raw_beta_first15pct,0.01462
train_raw_beta_last15pct,0.01115
train_weighted_beta_last15pct,0.22292


,respondents_seen,train_elapsed_sec,kriner_kl_loss,kriner_beta_mse,kriner_beta_mae,kriner_beta_mape_pct,spearman_rho,eval_prep_sec,eval_gpu_sec,eval_ols_ms,eval_wall_sec,eval_ms_per_task
step,,,,,,,,,,,,
0,0,0.00000,1.08491,0.01776,0.11429,133.52860,-0.26573,0.04481,8.95083,1.80457,10.71794,2.14359
29,464,59.36928,0.64091,0.00334,0.04460,69.15799,0.82517,0.00289,7.82433,1.99711,8.68216,1.73643
58,928,118.83610,0.62845,0.00348,0.05053,72.30867,0.84615,0.00291,7.81466,2.01392,8.65415,1.73083
87,1392,178.09251,0.64409,0.00403,0.05501,72.18531,0.83916,0.00292,7.81468,2.02528,8.64598,1.72920
116,1856,237.42315,0.63147,0.00329,0.04682,66.32093,0.83916,0.00295,7.82525,1.99593,8.67603,1.73521
117,1861,238.02826,0.63135,0.00329,0.04682,66.31061,0.83916,0.00288,7.81457,1.92750,8.62936,1.72587


In [17]:
# Compare Wu 2025 train metrics, held-out Kriner 2026 OLS/KL metrics, and timing metrics across completed runs
if not experiment_runs:
    raise ValueError("Run at least one `run_wu2025_epoch` cell first so `experiment_runs` is populated")
ran_keys = sorted(experiment_runs.keys())

compare_rows = [
    {
        "run": "step_0 (untrained)",
        "sample_size": 0,
        "beta_lambda": float("nan"),
        "num_epochs": 0,
        "steps_per_epoch": 0,
        "train_kl_last15pct": float("nan"),
        "train_raw_beta_last15pct": float("nan"),
        "kriner_kl_loss": step0_val_metrics["kriner_kl_loss"],
        "kriner_beta_mse": step0_val_metrics["kriner_beta_mse"],
        "kriner_beta_mae": step0_val_metrics["kriner_beta_mae"],
        "kriner_beta_mape_pct": step0_val_metrics["kriner_beta_mape_pct"],
        "spearman_rho": step0_val_metrics["spearman_rho"],
        "mean_step_ms": float("nan"),
        "mean_ms_per_respondent": float("nan"),
        "mean_eval_wall_sec": step0_val_metrics["eval_wall_sec"],
        "train_only_sec": 0.0,
        "eval_only_sec": step0_val_metrics["eval_wall_sec"],
        "total_wall_sec": step0_val_metrics["eval_wall_sec"],
    }
]
for s_size, lam in ran_keys:
    run = experiment_runs[(s_size, lam)]
    s = run["summary"]
    final_val = run["val_history_df"].iloc[-1]
    compare_rows.append(
        {
            "run": f"S={s_size}, lambda={lam:g}",
            "sample_size": s_size,
            "beta_lambda": lam,
            "num_epochs": int(s["num_epochs"]),
            "steps_per_epoch": int(s["steps_per_epoch"]),
            "train_kl_last15pct": float(s["train_kl_last15pct"]),
            "train_raw_beta_last15pct": float(s["train_raw_beta_last15pct"]),
            "kriner_kl_loss": float(final_val["kriner_kl_loss"]),
            "kriner_beta_mse": float(final_val["kriner_beta_mse"]),
            "kriner_beta_mae": float(final_val["kriner_beta_mae"]),
            "kriner_beta_mape_pct": float(final_val["kriner_beta_mape_pct"]),
            "spearman_rho": float(final_val["spearman_rho"]),
            "mean_step_ms": float(s["mean_step_ms"]),
            "mean_ms_per_respondent": float(s["mean_ms_per_respondent"]),
            "mean_eval_wall_sec": float(s["mean_eval_wall_sec"]),
            "train_only_sec": float(s["train_only_sec"]),
            "eval_only_sec": float(s["eval_only_sec"]),
            "total_wall_sec": float(s["total_wall_sec"]),
        }
    )

comparison_df = pd.DataFrame(compare_rows).set_index("run")
display(comparison_df.round(5))

# Compare the 12 predicted Kriner 2026 Table-2 AMCE betas against the ground-truth human Table-2 betas
kriner_betas_compare_df = pd.DataFrame(
    {
        "human_pop_beta": step0_beta_df["human_pop_beta"],
        "step_0": step0_beta_df["model_pop_beta"],
        **{
            f"S={s_size}, lambda={lam:g}": experiment_runs[(s_size, lam)]["final_beta_df"]["model_pop_beta"]
            for s_size, lam in ran_keys
        },
    }
)
display(kriner_betas_compare_df.round(4))

,sample_size,beta_lambda,steps_per_epoch,train_kl_last15pct,train_raw_beta_last15pct,kriner_kl_loss,kriner_beta_mse,kriner_beta_mae,kriner_beta_mape_pct,spearman_rho,mean_step_ms,mean_ms_per_respondent,mean_eval_wall_sec,train_only_sec,eval_only_sec,total_wall_sec
run,,,,,,,,,,,,,,,,
step_0 (untrained),0,NaN,0,NaN,NaN,1.08491,0.01776,0.11429,133.52860,-0.26573,NaN,NaN,10.71794,0.00000,10.71794,10.71794
"S=16, lambda=0",16,0.0,117,0.54015,0.01087,0.62682,0.00389,0.05537,75.39640,0.93706,2138.59724,134.45238,8.87200,250.21588,44.36002,294.97908
"S=16, lambda=1",16,1.0,117,0.53745,0.01020,0.62384,0.00294,0.04329,63.40753,0.89510,2040.62723,128.29306,8.71266,238.75339,43.56328,282.67510
"S=16, lambda=5",16,5.0,117,0.54127,0.01045,0.63163,0.00357,0.05084,67.46192,0.86014,2035.05325,127.94263,8.69338,238.10123,43.46688,281.90631
"S=16, lambda=20",16,20.0,117,0.54544,0.01115,0.63135,0.00329,0.04682,66.31061,0.83916,2034.42958,127.90342,8.65754,238.02826,43.28768,281.66353


,human_pop_beta,step_0,"S=16, lambda=0","S=16, lambda=1","S=16, lambda=5","S=16, lambda=20"
kriner_term,,,,,,
Airline: Frontier,-0.0854,0.0344,-0.0042,-0.0097,-0.0113,-0.0106
Airline: Southwest,-0.0238,0.0267,-0.0014,-0.0053,-0.0065,-0.0028
Airline: United,-0.0220,0.0010,0.0080,0.0048,0.0049,0.0036
Fuel: Sustainable aviation fuel,0.0691,-0.0574,0.0232,0.0299,0.0292,0.0272
Baggage: Checked bag+carry-on,0.0561,-0.0435,0.0051,-0.0004,-0.0002,-0.0031
Baggage: Personal item only,-0.1393,-0.0030,-0.0171,-0.0219,-0.0177,-0.0157
Space: Extra legroom,0.0653,-0.0813,0.0087,0.0473,0.0400,0.0365
Seat: Middle,-0.0906,-0.0057,-0.0104,-0.0043,0.0030,0.0018
Seat: Window,0.0384,0.0086,0.0038,0.0046,0.0084,0.0098
